In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#Load fine-tuned model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "FatalHub/qwen-function-calling-10k"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [5]:
# Testing the final model
prompt = """Tools:
[{"name": "live_giveaways_by_type", "description": "Retrieve live giveaways from the GamerPower API based on the specified type.", "parameters": {"type": {"description": "The type of giveaways to retrieve (e.g., game, loot, beta).", "type": "str", "default": "game"}}}]

Query:
Where can I find live giveaways for beta access and games?

Answer:

"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

print(tokenizer.decode(generated_tokens, skip_special_tokens=True))

[{"name": "live_giveaways_by_type", "arguments": {"type": "beta"}}, {"name": "live_giveaways_by_type", "arguments": {"type": "game"}}]


In [38]:
# template
prompt_template = """
Instructions:

You must follow the tool definitions exactly.

1. Only use tools that are provided in the Tools section.
2. For each selected tool, use only the parameters defined for that tool.
3. Do not invent, add, remove, rename, or modify any parameter.
4. Do not add parameters based on assumptions or information implied by the query.
5. Do not omit a parameter when it is required by the tool definition and its value can be determined from the query.
6. Preserve the exact parameter names and structure defined by the tool.
7. Use the values supported by the query and do not infer unsupported arguments.
8. Do not modify parameter values or types.
9. Return only the required tool calls in the expected JSON format.
10. Preserve the exact nesting and structure of tool arguments. Do not split fields that belong to the same object into separate objects or list elements.

Tools:

{tools}

Query:

{query}

Answer:
"""

# simple function to build the prompt
def build_prompt(tools, query):
    return prompt_template.format(
        tools=tools,
        query=query
    )

# pool of available tools
tools = """[
    {
        "name": "get_weather",
        "description": "Get the current weather for a specified city.",
        "parameters": {
            "city": {
                "description": "The city to get weather for.",
                "type": "str"
            }
        }
    },
    {
        "name": "search_restaurants",
        "description": "Search for restaurants in a specified city.",
        "parameters": {
            "city": {
                "description": "The city to search in.",
                "type": "str"
            },
            "cuisine": {
                "description": "The type of cuisine.",
                "type": "str",
                "default": "any"
            }
        }
    }
    {
        "name": "products_detail",
        "description": "Retrieve detailed information about a product.",
        "parameters": {
            "goods_id": {
                "description": "The product ID.",
                "type": "str"
            },
            "currency": {
                "description": "The currency in which the product information should be returned.",
                "type": "str"
            },
            "language": {
                "description": "The language for the product information.",
                "type": "str"
            }
        }
    },
    {
        "name": "search_products",
        "description": "Search for products using a search query.",
        "parameters": {
            "query": {
                "description": "The product search query.",
                "type": "str"
            }
        }
    },
    {
        "name": "get_product_reviews",
        "description": "Retrieve reviews for a product.",
        "parameters": {
            "goods_id": {
                "description": "The product ID.",
                "type": "str"
            }
        }
    }
    {
        "name": "dice_roll_probability",
        "description": "Calculate the probability of rolling a target sum using a specified number of dice with a specified number of faces.",
        "parameters": {
            "target_sum": {
                "description": "The target sum.",
                "type": "int"
            },
            "num_dice": {
                "description": "The number of dice.",
                "type": "int"
            },
            "num_faces": {
                "description": "The number of faces on each die.",
                "type": "int"
            }
        }
    },
    {
        "name": "simulate_query_database",
        "description": "Simulate a database query using a table and query conditions.",
        "parameters": {
            "table": {
                "description": "The database table to query.",
                "type": "str"
            },
            "conditions": {
                "conditions": {
                    "description": "A list containing one condition object. The condition object must contain all specified fields together. Comparison operators must use MongoDB-style objects
",
                    "type": "list",
                    "items": {
                        "type": "object"
    }
}
            }
        }
    },
    {
        "name": "airportstatistics",
        "description": "Retrieve airport statistics using an IATA airport code.",
        "parameters": {
            "iata": {
                "description": "The IATA airport code.",
                "type": "str"
            }
        }
    },
    {
        "name": "autocomplete",
        "description": "Autocomplete a search query for Flixbus stations.",
        "parameters": {
            "query": {
                "description": "The text to autocomplete.",
                "type": "str"
            }
        }
    },
    {
        "name": "get_random_challange",
        "description": "Retrieve a random drinking challenge of a specified type.",
        "parameters": {
            "type": {
                "description": "The type of challenge.",
                "type": "str"
            }
        }
    },
    {
        "name": "autocomplete_search",
        "description": "Search for movie autocomplete suggestions.",
        "parameters": {
            "search_value": {
                "description": "The movie title to search for.",
                "type": "str"
            },
            "search_type": {
                "description": "The type of autocomplete search : value of types 1=song, 2=match, 3=movie",
                "type": "int"
            }
        }
    }
]"""


In [39]:
import torch

In [40]:
query = "Simulate querying the 'sales' table with conditions: item='laptop', price>1000."
prompt = build_prompt(tools, query)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True,
    )

input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

[{"name": "simulate_query_database", "arguments": {"table": "sales", "conditions": [{"item": "laptop", "price": {"$gt": 1000}}]}}]


In [41]:
import json
import torch

test_cases = [
    {
        "query": "Retrieve information for product '75319' in Australian Dollars and Italian language.",
        "expected": [
            {
                "name": "products_detail",
                "arguments": {
                    "goods_id": "75319",
                    "currency": "AUD",
                    "language": "it"
                }
            }
        ]
    },

    {
        "query": "What are the chances of rolling exactly 20 with five 6-sided dice?",
        "expected": [
            {
                "name": "dice_roll_probability",
                "arguments": {
                    "target_sum": 20,
                    "num_dice": 5,
                    "num_faces": 6
                }
            }
        ]
    },

    {
        "query": "Simulate querying the 'sales' table with conditions: item='laptop', price>1000.",
        "expected": [
            {
                "name": "simulate_query_database",
                "arguments": {
                    "table": "sales",
                    "conditions": [
                        {
                            "item": "laptop",
                            "price": {
                                "$gt": 1000
                            }
                        }
                    ]
                }
            }
        ]
    },

    {
        "query": "What are the airport statistics for JFK airport today, and can you autocomplete 'New York' in English for Flixbus stations?",
        "expected": [
            {
                "name": "airportstatistics",
                "arguments": {
                    "iata": "JFK"
                }
            },
            {
                "name": "autocomplete",
                "arguments": {
                    "query": "New York"
                }
            }
        ]
    },

    {
        "query": "Fetch a funny drinking challenge and search for movies with the title 'The Great Escape'.",
        "expected": [
            {
                "name": "get_random_challange",
                "arguments": {
                    "type": "funny"
                }
            },
            {
                "name": "autocomplete_search",
                "arguments": {
                    "search_value": "The Great Escape",
                    "search_type": 3
                }
            }
        ]
    }
]


for i, case in enumerate(test_cases, 1):

    prompt = prompt_template.format(
        tools=tools,
        query=case["query"]
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            use_cache=True
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    print("=" * 80)
    print(f"CASE {i}")
    print("=" * 80)

    print("QUERY:")
    print(case["query"])

    print("\nMODEL OUTPUT:")
    print(response)

    print("\nEXPECTED:")
    print(json.dumps(case["expected"], indent=2))

    # Try to compare JSON automatically
    try:
        predicted = json.loads(response)

        if predicted == case["expected"]:
            print("\nRESULT: ✅ PASS")
        else:
            print("\nRESULT: ❌ FAIL")

    except json.JSONDecodeError:
        print("\nRESULT: ❌ FAIL (Invalid JSON)")

CASE 1
QUERY:
Retrieve information for product '75319' in Australian Dollars and Italian language.

MODEL OUTPUT:
[{"name": "products_detail", "arguments": {"goods_id": "75319", "currency": "AUD", "language": "it"}}]

EXPECTED:
[
  {
    "name": "products_detail",
    "arguments": {
      "goods_id": "75319",
      "currency": "AUD",
      "language": "it"
    }
  }
]

RESULT: ✅ PASS
CASE 2
QUERY:
What are the chances of rolling exactly 20 with five 6-sided dice?

MODEL OUTPUT:
[{"name": "dice_roll_probability", "arguments": {"target_sum": 20, "num_dice": 5, "num_faces": 6}}]

EXPECTED:
[
  {
    "name": "dice_roll_probability",
    "arguments": {
      "target_sum": 20,
      "num_dice": 5,
      "num_faces": 6
    }
  }
]

RESULT: ✅ PASS
CASE 3
QUERY:
Simulate querying the 'sales' table with conditions: item='laptop', price>1000.

MODEL OUTPUT:
[{"name": "simulate_query_database", "arguments": {"table": "sales", "conditions": [{"item": "laptop", "price": {"$gt": 1000}}]}}]

EXPECTED:

All cases that failed previously [5/5] are solved by :
- more training
- prompt Engineering
- good tool description